# M3L4 E12 — Ciclo completo de mejora iterativa
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

| Módulo | Concepto | Por qué lo necesitas acá |
|---|---|---|
| M3L4 E04-E05 | Golden dataset, evaluate_router, comparación v1 vs v2 | Este ejercicio unifica todo el flujo de mejora |
| M3L4 E07 | Dashboard de métricas (accuracy, fallback_rate, etc.) | Las métricas del dashboard guían las decisiones |
| M3L4 E03 | Diagnosticar patrones de falla | Identificar causa raíz de los fallos del router |
| M3L4 E00-E02 | Tracing como herramienta de observabilidad | El ciclo usa tracing para detectar problemas |
| Python | pandas DataFrames, groupby | Para análisis de resultados y comparación |

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Ciclo de mejora** | Proceso iterativo de medir -> diagnosticar -> corregir -> medir de nuevo | Los 8 pasos del ciclo documentados al inicio |
| **Causa raíz** | Problema fundamental que causa los fallos (ej: keywords faltantes) | Análisis de `df_failures_v1` para ver patrones |
| **Fix** | Cambio específico para resolver la causa raíz | Router v2 con vocabulario expandido |
| **Evidencia cuantitativa** | Métricas antes/después que demuestran la mejora | `acc_v1` vs `acc_v2` en tabla de comparación |
| **Reporte de mejora** | Documentación del cambio: issue, causa, fix, métricas | Reporte Markdown generado al final |

---

## El ciclo

```
1. Instrumentar (E00-E02: MiniTracer)
       |
2. Ejecutar golden dataset (v1)  <--- (E04, E05)
       |
3. Observar trazas -> detectar patrón de fallo  <--- (E03, E07)
       |
4. Priorizar fix
       |
5. Implementar cambio (router v2)  <--- (E05)
       |
6. Re-ejecutar golden dataset
       |
7. Comparar métricas antes vs después  <--- (E05, E07)
       |
8. Documentar el cambio  <--- (ESTE EJERCICIO)
```

**Objetivo del ejercicio:** ejecutar el ciclo completo production-grade: detectar un problema, aplicar un fix, medir el impacto con evidencia.

## Escenario de este ejercicio

> **Problema detectado:** las consultas de facturación/reembolsos se están yendo a `GeneralAgent` porque el router v1 no reconoce términos como "reembolso", "gasto" y "comprobante".

**Tu tarea:** reproducir el diagnóstico, aplicar el fix y documentarlo.

## Setup

| Import | Qué hace |
|---|---|
| `pandas` | DataFrames para evaluación y comparación |

```python
!pip install pandas -q
import pandas as pd
```

In [ ]:
!pip install pandas -q
import pandas as pd
print('Setup listo.')

## Paso 1 — Router v1 (con el bug)

Router mínimo con palabras clave limitadas. No reconoce sinónimos como "licencia", "reembolso" o "comprobante".

| Palabra clave | Intent |
|---|---|
| `'vacaciones'` | `hr` |
| `'vpn'` o `'error'` | `it` |
| `'factura'` | `finance` |
| `'contrato'` | `legal` |
| *ninguna* | `general` |

**Bug conocido:** queries de finance que usan "reembolso" o "comprobante" caen en `general`. Queries de HR que usan "licencia" o "recibo" también.

In [ ]:
def route_query_v1(query: str) -> str:
    """Router v1: reglas básicas con vocabulario limitado."""
    q = query.lower()
    if 'vacaciones' in q:
        return 'hr'
    if 'vpn' in q or 'error' in q:
        return 'it'
    if 'factura' in q:
        return 'finance'
    if 'contrato' in q:
        return 'legal'
    return 'general'

print('Router v1 listo.')

## Paso 2 — Golden Dataset

Dataset de 12 casos que expone las debilidades de v1:

- `case_007`: "reembolso de gastos" (finance) -> v1 no reconoce
- `case_009`: "licencia por enfermedad" (hr) -> v1 no reconoce
- `case_010`: "comprobante de pago de mi salario" (finance) -> v1 no reconoce
- `case_012`: "ok" (clarification) -> v1 no tiene regla para queries cortas

In [ ]:
golden_dataset = [
    {'id': 'case_001', 'query': 'Cómo solicito mis días de vacaciones?',          'expected_intent': 'hr'},
    {'id': 'case_002', 'query': 'Mi VPN no conecta desde ayer',                     'expected_intent': 'it'},
    {'id': 'case_003', 'query': 'Necesito ver mi factura del mes pasado',            'expected_intent': 'finance'},
    {'id': 'case_004', 'query': 'Necesito el contrato de confidencialidad actualizado', 'expected_intent': 'legal'},
    {'id': 'case_005', 'query': 'No puedo entrar al portal para ver mi recibo',     'expected_intent': 'hr'},
    {'id': 'case_006', 'query': 'ayuda',                                             'expected_intent': 'clarification'},
    {'id': 'case_007', 'query': 'Cuándo se procesa el reembolso de gastos?',       'expected_intent': 'finance'},
    {'id': 'case_008', 'query': 'El sistema de login no me deja entrar',             'expected_intent': 'it'},
    {'id': 'case_009', 'query': 'Quiero pedir una licencia por enfermedad',          'expected_intent': 'hr'},
    {'id': 'case_010', 'query': 'Puedo ver el comprobante de pago de mi salario?', 'expected_intent': 'finance'},
    {'id': 'case_011', 'query': 'Necesito firmar un NDA con un proveedor externo',  'expected_intent': 'legal'},
    {'id': 'case_012', 'query': 'ok',                                                'expected_intent': 'clarification'},
]
print(f'{len(golden_dataset)} casos.')

## Paso 3 — TODO: Función de evaluación

Misma función `evaluate_router()` de E04/E05. Recibe un router y un dataset, devuelve `(df, accuracy)`.

In [ ]:
def evaluate_router(router_fn, dataset):
    # TODO: misma función de E04/E05
    pass

print('Función definida.')

## Paso 4 — Ejecutar Router v1 y diagnosticar

Evaluamos v1 para ver el estado actual y detectar los patrones de fallo.

In [ ]:
# TODO: evaluar router v1
df_v1, acc_v1 = None, None
print(f'Router v1 accuracy: {acc_v1:.2%}' if acc_v1 else 'TODO')

In [ ]:
# TODO: mostrar los casos que fallaron con v1
# Qué tienen en común?
df_failures_v1 = None
df_failures_v1

## Paso 5 — Diagnóstico

Completa el análisis de causa raíz basado en los casos que fallaron.

**Causa raíz identificada:**

> TODO: listar qué keywords faltan en router v1 que causan los fallos de finance/hr

**Queries que fallan:**
- `case_007`: "Cuándo se procesa el **reembolso** de **gastos**?" -> router v1 no reconoce 'reembolso' ni 'gastos'
- `case_010`: "Puedo ver el **comprobante** de pago de mi **salario**?" -> mismo problema
- `case_005`: "portal para ver mi **recibo**" -> 'recibo' no está en las keywords de hr de v1
- `case_009`: "licencia por enfermedad" -> 'licencia' no está en v1
- `case_012`: "ok" -> debería ser clarification pero v1 devuelve general

## Paso 6 — TODO: Implementar Router v2 con el fix

Router v2 debe:

1. **Expandir vocabulario de HR:** agregar 'licencia', 'recibo', 'nómina', 'beneficios', 'rrhh', 'portal rrhh'
2. **Expandir vocabulario de IT:** agregar 'app', 'laptop', 'wifi', 'login', 'contraseña', 'sistema', 'acceso'
3. **Expandir vocabulario de Finance:** agregar 'pago', 'reembolso', 'gasto', 'cobro', 'comprobante', 'salario'
4. **Expandir vocabulario de Legal:** agregar 'confidencialidad', 'nda', 'acuerdo', 'firma'
5. **Detectar multi-intent:** si hay keywords de 2+ dominios, devolver 'multi_intent'
6. **Detectar clarification:** queries de <= 2 palabras

In [ ]:
def route_query_v2(query: str) -> str:
    """
    Router v2: vocabulario expandido y detección de multi-intent/clarification.

    Agrega sobre v1:
    - hr: licencia, recibo, nómina, beneficios, rrhh
    - it: app, laptop, wifi, login, contraseña, sistema, acceso
    - finance: pago, reembolso, gasto, cobro, comprobante, salario
    - legal: confidencialidad, nda, acuerdo, firma
    - multi_intent: cuando detecta 2+ dominios
    - clarification: queries de <= 2 palabras
    """
    # TODO: implementar v2
    pass

print('Router v2 definido.')

## Paso 7 — Comparar resultados

Evaluamos v2 y comparamos métricas con v1.

In [ ]:
# TODO: evaluar v2 y comparar con v1
df_v2, acc_v2 = None, None

if acc_v1 and acc_v2:
    print(f'Router v1 accuracy: {acc_v1:.2%}')
    print(f'Router v2 accuracy: {acc_v2:.2%}')
    print(f'Mejora: +{(acc_v2 - acc_v1):.2%}')

In [ ]:
# TODO: tabla de comparación
if acc_v1 and acc_v2:
    comparison = pd.DataFrame([
        {'version': 'router_v1', 'routing_accuracy': acc_v1},
        {'version': 'router_v2', 'routing_accuracy': acc_v2}
    ])
    comparison

## Paso 8 — TODO: Generar reporte de mejora

Documenta el cambio completo: problema detectado, causa raíz, fix aplicado, métricas antes/después y riesgos pendientes.

In [ ]:
report = f"""
# Fix Report - Router Finance/HR Misclassification

## Issue
TODO: describir el problema detectado

## Causa raíz
TODO: listar los términos faltantes y por qué fallaban

## Cambio aplicado
TODO: describir qué se agregó en v2

## Métricas
| Version | Routing Accuracy |
|---|---:|
| router_v1 | TODO |
| router_v2 | TODO |

## Validación
El cambio fue validado usando golden_dataset_v1 ({len(golden_dataset)} casos).

## Riesgo pendiente
TODO: qué puede seguir fallando?
"""
print(report)

In [ ]:
assert acc_v1 is not None, 'acc_v1 es None'
assert acc_v2 is not None, 'acc_v2 es None'
assert acc_v2 >= acc_v1, f'v2 ({acc_v2:.2%}) debería ser >= v1 ({acc_v1:.2%})'

# Verificar que v2 corrige los casos de finance
df_v2_ok, _ = evaluate_router(route_query_v2, [c for c in golden_dataset if c['expected_intent'] == 'finance'])
finance_acc_v2 = df_v2_ok['correct'].mean()
assert finance_acc_v2 > 0, 'Finance debería tener al menos algunos aciertos en v2'

print(f'Checks E12 OK')
print(f'v1: {acc_v1:.2%} -> v2: {acc_v2:.2%} (+{(acc_v2-acc_v1):.2%})')
print(f'Finance accuracy v2: {finance_acc_v2:.2%}')

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| v2 no mejora a v1 | El fix no agrega las keywords correctas | `acc_v2 <= acc_v1` |
| Finance sigue fallando en v2 | No agregar 'reembolso', 'gasto', 'comprobante', 'salario' | `finance_acc_v2 == 0` |
| Reporte sin evidencias | No incluir métricas numéricas en el reporte | El reporte tiene TODOs sin completar |
| No documentar riesgos | El reporte no menciona qué puede seguir fallando | Falta la sección de riesgo pendiente |
| No evaluar antes del fix | No tener `acc_v1` para comparar | No se puede medir la mejora |

## Síntesis

### El ciclo completo

```
1. Detectar problema (E03, E07)
   |  "Finance tiene accuracy baja"
   v
2. Diagnosticar causa raíz (E12)
   |  "Faltan keywords: reembolso, gasto, comprobante"
   v
3. Implementar fix (E12)
   |  "Router v2 con vocabulario expandido"
   v
4. Medir impacto (E04, E05, E12)
   |  "v1: 58% -> v2: 92%"
   v
5. Documentar (E12)
   |  "Reporte de mejora con evidencias"
   v
6. Repetir!
```

### Qué construiste en M3L4

| Ejercicio | Aporte al ciclo |
|---|---|
| **E00** | Concepto de trace vs log |
| **E01-E02** | MiniTracer: instrumentación manual |
| **E03** | Diagnóstico automático de fallas |
| **E04** | Golden dataset para medir accuracy |
| **E05** | Comparación de versiones |
| **E06** | Evaluación de calidad de respuestas |
| **E07** | Dashboard unificado de métricas |
| **E08-E10** | LangGraph + Langfuse: tracing automático |
| **E11** | Scores en Langfuse para tracking |
| **E12** | Ciclo completo de mejora iterativa |